This is a program which is in testing phase. We will use a linear regression ML model to give an SOC number for the flights.

In [1]:
# Set the testing flights and training flight
train_flight_one_id = 4620
train_flight_two_id = 4940
test_flight_id = 4929

In [ ]:
# Import the querying module
from flight_querying import query_flights
import pandas as pd

# Set up and retrieve the data from the database.
db_connect = query_flights()
train_one_data = db_connect.connect_flight_for_ml(train_flight_one_id)
train_two_data = db_connect.connect_flight_for_ml(train_flight_two_id)

# Set up train data
train_x = pd.concat([train_one_data, train_two_data], axis=0)
train_y = train_x["SOC"].to_numpy()
numeric_train_x = train_x.drop(columns=["Operations", "SOC"])

# Set up test data
test_x = db_connect.connect_flight_for_ml(test_flight_id)
test_y = test_x["SOC"].to_numpy()
numeric_test_x = test_x.drop(columns=["Operations", "SOC"])

One-Hot-Encoding of the Operations columns

In [ ]:
# ONE-HOT ENCODE
# https://stackabuse.com/one-hot-encoding-in-python-with-pandas-and-scikit-learn/
def one_hot(df, col, pre):
  encoded = pd.get_dummies(df[col], prefix=pre)
  for column in encoded:
    encoded = encoded.rename(columns={column: col + "_" + column})
  encoded['Time'] = df['Time']
  return encoded

In [ ]:
# Encode Train data
train_encoded = one_hot(train_x, "Operations", 'is')
numeric_train_x = pd.merge(numeric_train_x, train_encoded, on='Time', how='left')

# Encode Test data
test_encoded = one_hot(test_x, "Operations", 'is')
numeric_test_x = pd.merge(numeric_test_x, test_encoded, on='Time', how='left')

Machine Learning Model Implementation

In [ ]:
# import sklearn
from sklearn import preprocessing, svm
from sklearn.linear_model import LinearRegression

# Set model
regression_model = LinearRegression()

# Fit model
regression_model.fit(numeric_train_x, train_y)

# print model score
print(regression_model.score(numeric_test_x, test_y))